# 08 · Verifikasi implementasi dan determinisme

**Bab 3 Subbab 3.12.** Notebook ini memastikan kode notebook benar, bukan memilih hasil.

1. **Fitur dan label** dibandingkan dengan pipeline penelitian lama (`src/thesis_experiments`) yang sudah teruji
   197 tes. Perbedaan yang diharapkan hanya pada aturan yang sengaja diubah: pemanasan (bar ke-50 vs bar ke-51).
2. **Fitur makro** dibandingkan dengan `add_macro_features` lama.
3. **Purge** dibandingkan dengan aturan lama (batas = awal blok).
4. **Determinisme**: satu periode dilatih ulang, prediksinya harus identik.
5. **Pembanding** skor dengan run lama 21–22 September (hanya informasi; run lama memakai aturan pemanasan berbeda).

In [1]:
import tomllib
from pathlib import Path

# Cari akar repositori (folder yang memuat pyproject.toml) agar notebook bisa dijalankan dari mana saja.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
CFG = tomllib.loads((ROOT / "configs/phase4-p0.toml").read_text(encoding="utf-8"))
OUT = ROOT / "data/interim/phase4-p0"     # folder hasil; diabaikan Git karena memuat data vendor
OUT.mkdir(parents=True, exist_ok=True)

In [2]:
import importlib
import sys

import numpy as np
import pandas as pd

sys.path.insert(0, str(ROOT / "src"))
add_macro_features = importlib.import_module("thesis_experiments.macro_features").add_macro_features

baru = pd.read_parquet(OUT / "05_dataset.parquet")
label = pd.read_parquet(OUT / "02_label.parquet")
fitur = pd.read_parquet(OUT / "03_fitur.parquet")
semua = label.merge(fitur.drop(columns="sigma_24"), on="time")
LAMA_FITUR = ROOT / "data/interim/phase4/full-data-features/metaquotes-demo/primary/features-threshold-0p5.parquet"
LAMA_PRED = ROOT / "data/interim/phase4/model-results/metaquotes-demo-primary-0p5-k3/predictions.parquet"
ADA_RUN_LAMA = LAMA_FITUR.exists() and LAMA_PRED.exists()   # hanya ada di repo penelitian
lama = pd.read_parquet(LAMA_FITUR) if ADA_RUN_LAMA else None
print("Baris notebook:", len(semua), "· artefak run lama tersedia:", ADA_RUN_LAMA)

Baris notebook: 38799 · artefak run lama tersedia: True


## 1. Fitur K1/K2, waktu, segmen, dan label

In [3]:
if ADA_RUN_LAMA:
    K2 = CFG["fitur"]["k1"] + CFG["fitur"]["teknikal"]
    g = semua.merge(lama, on="time", suffixes=("", "_lama"), validate="1:1")
    assert len(g) == len(semua) == len(lama)

    banding = {}
    for k in K2:
        a, b = g[k].to_numpy(float), g[f"{k}_lama"].to_numpy(float)
        sama_nan = np.isnan(a) == np.isnan(b)
        banding[k] = {"posisi NaN sama": bool(sama_nan.all()), "selisih maks": float(np.nanmax(np.abs(a - b)))}
    display(pd.DataFrame(banding).T)
else:
    print("Artefak run lama tidak ada di salinan ini; bagian ini hanya berjalan di repo penelitian.")

,posisi NaN sama,selisih maks
return_lag_0,True,0.0
return_lag_1,True,0.0
return_lag_2,True,0.0
return_lag_3,True,0.0
return_lag_4,True,0.0
return_lag_5,True,0.0
log_close_open,True,0.0
log_high_low,True,0.0
sigma_24,True,0.0
rsi_14,True,0.0


In [4]:
if ADA_RUN_LAMA:
    cek = {
        "decision_time sama": (g["decision_time"] == g["decision_time_lama"]).all(),
        "target_end_time sama (bar layak)": (g.loc[g["target_layak"], "target_end_time"] == g.loc[g["target_layak"], "target_end_time_lama"]).all(),
        "segment_id sama": (g["segment_id"] == g["segment_id_lama"]).all(),
        "target_layak sama": (g["target_layak"] == g["target_eligible"]).all(),
        "label 0,5 sama (bar layak)": (g.loc[g["target_layak"], "label_0p5"] == g.loc[g["target_layak"], "target_label"]).all(),
    }
    display(pd.Series(cek, name="lulus").to_frame())
else:
    print("Artefak run lama tidak ada di salinan ini; bagian ini hanya berjalan di repo penelitian.")

,lulus
decision_time sama,True
target_end_time sama (bar layak),True
segment_id sama,True
target_layak sama,True
"label 0,5 sama (bar layak)",True


### Perbedaan kelayakan fitur hanya karena aturan pemanasan

In [5]:
if ADA_RUN_LAMA:
    beda = g[g["fitur_layak"] != g["feature_eligible"]]
    print("Baris berbeda:", len(beda), "· semuanya bar ke-", sorted(beda["bar_ke"].unique()), " dalam segmen", sep="")
    assert set(beda["bar_ke"]) == {50} and (~beda["fitur_layak"] & beda["feature_eligible"]).all()
else:
    print("Artefak run lama tidak ada di salinan ini; bagian ini hanya berjalan di repo penelitian.")

Baris berbeda:9· semuanya bar ke-[np.int64(50)] dalam segmen


## 2. Fitur makro dibandingkan dengan pipeline lama

In [6]:
kal = pd.read_csv(ROOT / CFG["inputs"]["kalender"]["path"])
kal["event_time_utc"] = pd.to_datetime(kal["event_time_server"], format="%Y.%m.%dT%H:%M:%S", utc=True)
ref = add_macro_features(label[["time", "decision_time"]], kal)
makro = pd.read_parquet(OUT / "04_fitur_makro.parquet")
kolom = [c for c in makro.columns if c.startswith("macro_")]
assert list(ref[kolom].columns) == kolom
a, b = makro[kolom].to_numpy(float), ref[kolom].to_numpy(float)
print("Kolom makro:", len(kolom), "· posisi NaN sama:", bool((np.isnan(a) == np.isnan(b)).all()),
      "· selisih maks:", float(np.nanmax(np.abs(a - b))))

Kolom makro: 46 · posisi NaN sama: True · selisih maks: 0.0


## 3. Purge: aturan Bab 3 dibandingkan aturan lama

Aturan Bab 3: data latih = akhir label < waktu keputusan pertama blok. Aturan lama: akhir label < awal blok.

In [7]:
blok = pd.read_csv(OUT / "05_blok.csv", parse_dates=["awal", "akhir", "keputusan_pertama"])
blok["n_latih_aturan_lama"] = [int((baru["target_end_time"] < a).sum()) for a in blok["awal"]]
blok["selisih"] = blok["n_latih"] - blok["n_latih_aturan_lama"]
blok[["periode", "blok", "awal", "keputusan_pertama", "n_latih", "n_latih_aturan_lama", "selisih"]]

,periode,blok,awal,keputusan_pertama,n_latih,n_latih_aturan_lama,selisih
0,2023-H1,validasi-1,2021-07-01 00:00:00+00:00,2021-07-01 02:00:00+00:00,8279,8278,1
1,2023-H1,validasi-2,2022-01-01 00:00:00+00:00,2022-01-03 02:00:00+00:00,11093,11093,0
2,2023-H1,validasi-3,2022-07-01 00:00:00+00:00,2022-07-01 02:00:00+00:00,13901,13900,1
3,2023-H1,uji,2023-01-01 00:00:00+00:00,2023-01-03 02:00:00+00:00,16752,16752,0
4,2023-H2,validasi-1,2022-01-01 00:00:00+00:00,2022-01-03 02:00:00+00:00,11093,11093,0
5,2023-H2,validasi-2,2022-07-01 00:00:00+00:00,2022-07-01 02:00:00+00:00,13901,13900,1
6,2023-H2,validasi-3,2023-01-01 00:00:00+00:00,2023-01-03 02:00:00+00:00,16752,16752,0
7,2023-H2,uji,2023-07-01 00:00:00+00:00,2023-07-03 02:00:00+00:00,19560,19559,1
8,2024-H1,validasi-1,2022-07-01 00:00:00+00:00,2022-07-01 02:00:00+00:00,13901,13900,1
9,2024-H1,validasi-2,2023-01-01 00:00:00+00:00,2023-01-03 02:00:00+00:00,16752,16752,0


## 4. Determinisme: latih ulang satu periode

Mengulang penyetelan dan pelatihan XGBoost K2 periode 2023-H1 pada ambang 0,5 dengan kode notebook 06, lalu
membandingkan prediksinya dengan hasil run.

In [8]:
import nbformat

lokasi_06 = [Path("06_pelatihan_model.ipynb"), ROOT / "notebooks/fase4/06_pelatihan_model.ipynb"]
sumber = nbformat.read(next(p for p in lokasi_06 if p.exists()), as_version=4)
ruang = {}
for sel in sumber.cells:            # jalankan sel kode sampai definisi fungsi penyetelan
    if sel.cell_type == "code":
        exec(sel.source, ruang)  # noqa: S102 (menjalankan kode notebook 06 sendiri)
        if "def setel(" in sel.source:
            break

catatan = []
latih, uji = ruang["potong"]("2023-H1", "uji")
param = ruang["setel"]("xgboost", ruang["K2"], "label_0p5", "2023-H1", catatan)
ulang, _ = ruang["latih_prediksi"]("xgboost", param, latih, uji, ruang["K2"], "label_0p5")

run = pd.read_parquet(OUT / "06_prediksi.parquet")
run = run[(run["ambang"] == 0.5) & (run["periode"] == "2023-H1") & (run["model"] == "xgboost")
          & (run["konfigurasi"] == "K2") & (run["analisis"] == "disetel")]
print("Parameter terpilih ulang:", param, "· pada run:", run["parameter"].iloc[0])
print("Prediksi identik:", bool((run["prediksi"].to_numpy() == ulang).all()), f"({len(ulang)} bar)")
assert (run["prediksi"].to_numpy() == ulang).all()

Konfigurasi: {'K1': 9, 'K2': 13, 'K3': 59} · status K3: LAYAK


Parameter terpilih ulang: {'max_depth': 3, 'n_estimators': 100, 'learning_rate': 0.05} · pada run: {'max_depth': 3, 'n_estimators': 100, 'learning_rate': 0.05}
Prediksi identik: True (2807 bar)


## 5. Pembanding dengan run lama (informasi)

Run lama 21–22 September memakai pemanasan bar ke-50 dan purge berbatas awal blok. Selisih skor menunjukkan seberapa
besar pengaruh dua perbedaan aturan itu. Angka ini **bukan** dasar untuk memilih aturan.

In [9]:
from sklearn.metrics import f1_score

if ADA_RUN_LAMA:
    lama_pred = pd.read_parquet(LAMA_PRED)
    lama_pred = lama_pred[lama_pred["analysis"] == "tuned"]
    lama_skor = (lama_pred.groupby(["fold", "model", "configuration"])[["y_true", "y_pred"]]
                 .apply(lambda d: f1_score(d["y_true"], d["y_pred"], labels=["SELL", "HOLD", "BUY"], average="macro", zero_division=0))
                 .rename("macro_f1_lama").reset_index()
                 .rename(columns={"fold": "periode", "configuration": "konfigurasi"}))
    lama_skor["model"] = lama_skor["model"].replace({"logistic_regression": "regresi_logistik"})

    metrik = pd.read_csv(OUT / "07_metrik_per_periode.csv")
    baru_skor = metrik[(metrik["ambang"] == 0.5) & (metrik["analisis"] == "disetel") & (metrik["model"] != "mayoritas")]
    b = baru_skor.merge(lama_skor, on=["periode", "model", "konfigurasi"])
    b["selisih"] = b["macro_f1"] - b["macro_f1_lama"]
    print("Pasangan:", len(b), "· selisih absolut maks:", round(b["selisih"].abs().max(), 6))
    display(b.groupby(["model", "konfigurasi"])[["macro_f1", "macro_f1_lama"]].mean().round(6))
else:
    print("Artefak run lama tidak ada di salinan ini; bagian ini hanya berjalan di repo penelitian.")

Pasangan: 42 · selisih absolut maks: 0.010899


macro_f1  macro_f1_lama
model            konfigurasi                         
regresi_logistik K1           0.392154       0.392300
                 K2           0.392243       0.392374
                 K3           0.391609       0.390922
xgboost          K1           0.393499       0.390178
                 K2           0.392416       0.393466
                 K3           0.392895       0.393791